In [ ]:
import secret

### try crypto.com
#### (not working)

In [ ]:
import hashlib
import hmac
import time
from rich import print
import requests

API_KEY = secret.api_key
SECRET_KEY = secret.secret_key

def private_request(method: str, params: dict | None = None) -> dict:
    params = params or {}

    nonce = str(int(time.time() * 1000))
    request_id = nonce

    param_str = "".join(f"{k}{params[k]}" for k in sorted(params))

    payload = f"{method}{request_id}{API_KEY}{param_str}{nonce}"

    sig = hmac.new(
        SECRET_KEY.encode(),
        payload.encode(),
        hashlib.sha256,
    ).hexdigest()

    body = {
        "id": request_id,
        "method": method,
        "api_key": API_KEY,
        "sig": sig,
        "nonce": nonce,
        "params": params,
    }

    return requests.post(
        f"https://api.crypto.com/exchange/v1/{method}",
        json=body,
        timeout=10,
    ).json()


print(private_request("private/get-accounts"))
accounts = private_request("private/get-accounts")

for x in accounts["result"]["data"]:
    if float(x["balance"]) > 0:
        print(x["currency"], x["balance"], x["available"])

### try kraken api without api key.
* this should tell if the url works

In [ ]:
import requests
from rich import print

url = "https://api.kraken.com/0/public/Ticker"
data = requests.get(url, params={"pair": "BTCUSD"}, timeout=10).json()

print(data)

### request current funds

In [6]:
import base64, hashlib, hmac, time, urllib.parse, requests
from typing import Literal

API_KEY = secret.kraken_api_key
API_SECRET = secret.kraken_api_secret


def private_request(endpoint: Literal[
                    "Balance",
                    "AddOrder",
                    "OpenOrders",
                    "ClosedOrders",
                    "CancelOrder"], 
                    data: dict | None = None) -> dict:

    data = {} if data is None else data
    data["nonce"] = str(time.time_ns())

    path = f"/0/private/{endpoint}"
    encoded = urllib.parse.urlencode(data)

    signature = base64.b64encode(
        hmac.new(
            base64.b64decode(API_SECRET),
            path.encode() + hashlib.sha256((data["nonce"] + encoded).encode()).digest(),
            hashlib.sha512,
        ).digest()
    ).decode()

    return requests.post(
        "https://api.kraken.com" + path,
        data=data,
        headers={"API-Key": API_KEY, "API-Sign": signature},
        timeout=10,
    ).json()


print(private_request("Balance"))

{'error': [], 'result': {'ZCAD': '5.0000'}}

In [7]:
order = private_request(
    "AddOrder",
    {
        "ordertype": "limit",
        "type": "buy",
        "pair": "XBTUSD",
        "price": "10000",
        "volume": "0.0001",
        "validate": "false",
    },
)

print(order)

{'error': ['EOrder:Insufficient funds']}

## kraken class for trading

In [10]:
import base64, hashlib, hmac, time, urllib.parse, requests, secret
from decimal import Decimal
from typing import Any, Literal

PrivateEndpoint = Literal[
    "Balance",
    "OpenOrders",
    "ClosedOrders",
    "QueryOrders",
    "TradesHistory",
    "AddOrder",
    "AmendOrder",
    "CancelOrder",
    "CancelAll",
    "CancelAllOrdersAfter",
]

PublicEndpoint = Literal[
    "Ticker",
    "Depth",
    "AssetPairs",
]

Side = Literal["buy", "sell"]
TimeInForce = Literal["GTC", "IOC", "GTD", "FOK"]
Number = str | int | float | Decimal


class Kraken:
    def __init__( self, api_key: str, api_secret: str, timeout: float = 10.0) -> None:
        self.api_key = api_key
        self.api_secret = api_secret
        self.timeout = timeout
        self.url = "https://api.kraken.com"
        self.session = requests.Session()
        self._last_nonce = 0

    @staticmethod
    def _str(value: Number) -> str:
        return format(value, "f") if isinstance(value, Decimal) else str(value)

    def _nonce(self) -> str:
        nonce = max(time.time_ns(), self._last_nonce + 1)
        self._last_nonce = nonce
        return str(nonce)

    @staticmethod
    def _result(response: requests.Response) -> dict[str, Any]:
        response.raise_for_status()
        data = response.json()

        if data["error"]:
            raise RuntimeError(" | ".join(data["error"]))

        return data["result"]

    def _public(self, endpoint: PublicEndpoint, data: dict[str, Any] | None = None) -> dict[str, Any]:
        response = self.session.get(
            f"{self.url}/0/public/{endpoint}",
            params=data,
            timeout=self.timeout,
        )
        return self._result(response)

    def _private(self, endpoint: PrivateEndpoint, data: dict[str, Any] | None = None) -> dict[str, Any]:
        data = {} if data is None else data.copy()

        nonce = self._nonce()
        data["nonce"] = nonce

        path = f"/0/private/{endpoint}"
        encoded = urllib.parse.urlencode(data)

        message = (
            path.encode()
            + hashlib.sha256((nonce + encoded).encode()).digest()
        )

        signature = base64.b64encode(
            hmac.new(
                base64.b64decode(self.api_secret),
                message,
                hashlib.sha512,
            ).digest()
        ).decode()

        response = self.session.post(
            self.url + path,
            data=encoded,
            headers={
                "API-Key": self.api_key,
                "API-Sign": signature,
                "Content-Type": "application/x-www-form-urlencoded",
            },
            timeout=self.timeout,
        )

        return self._result(response)

    # ---------- Market data ----------

    def ticker(self, pair: str) -> dict[str, Any]:
        return self._public("Ticker", {"pair": pair})

    def order_book(self, pair: str, count: int = 10) -> dict[str, Any]:
        return self._public(
            "Depth",
            {
                "pair": pair,
                "count": count,
            },
        )

    def pair_info(self, pair: str) -> dict[str, Any]:
        return self._public("AssetPairs", {"pair": pair})

    # ---------- Account ----------

    def balance(self) -> dict[str, Any]:
        return self._private("Balance")

    def open_orders(self) -> dict[str, Any]:
        return self._private("OpenOrders")

    def closed_orders(self) -> dict[str, Any]:
        return self._private("ClosedOrders")

    def order(self, txid: str) -> dict[str, Any]:
        return self._private("QueryOrders", {"txid": txid})

    def trades(self) -> dict[str, Any]:
        return self._private("TradesHistory")

    # ---------- Orders ----------

    def limit_order(self, side: Side, pair: str, volume: Number, price: Number, 
                    *, validate: bool = False, post_only: bool = False, time_in_force: TimeInForce = "GTC") -> dict[str, Any]:
        data: dict[str, Any] = dict(
            pair = pair,
            type = side,
            ordertype = "limit",
            volume = self._str(volume),
            price = self._str(price),
            timeinforce = time_in_force,
            validate = str(validate).lower()
        )

        if post_only:
            data["oflags"] = "post"

        return self._private("AddOrder", data)

    def market_order(self, side: Side, pair: str, volume: Number, *, validate: bool = False) -> dict[str, Any]:
        return self._private(
            "AddOrder",
            {
                "pair": pair,
                "type": side,
                "ordertype": "market",
                "volume": self._str(volume),
                "validate": str(validate).lower(),
            },
        )

    def buy_limit(self, pair: str, volume: Number, price: Number, **kwargs: Any) -> dict[str, Any]:
        return self.limit_order("buy", pair, volume, price, **kwargs)

    def sell_limit(self, pair: str, volume: Number, price: Number, **kwargs: Any) -> dict[str, Any]:
        return self.limit_order("sell", pair, volume, price, **kwargs)

    def buy_market(self, pair: str, volume: Number, *, validate: bool = False) -> dict[str, Any]:
        return self.market_order("buy", pair, volume, validate=validate)

    def sell_market(self, pair: str, volume: Number, *, validate: bool = False) -> dict[str, Any]:
        return self.market_order("sell", pair, volume, validate=validate)

    # ---------- Modify / cancel ----------

    def amend(self, txid: str, *, price: Number | None = None, volume: Number | None = None, post_only: bool = False) -> dict[str, Any]:
        data: dict[str, Any] = {"txid": txid}

        if price is not None:
            data["limit_price"] = self._str(price)

        if volume is not None:
            data["order_qty"] = self._str(volume)

        if post_only:
            data["post_only"] = "true"

        return self._private("AmendOrder", data)

    def cancel(self, txid: str) -> dict[str, Any]:
        return self._private("CancelOrder", {"txid": txid})

    def cancel_all(self) -> dict[str, Any]:
        return self._private("CancelAll")

    def cancel_after(self, seconds: int) -> dict[str, Any]:
        return self._private(
            "CancelAllOrdersAfter",
            {"timeout": seconds},
        )

    def close(self) -> None:
        self.session.close()

api = Kraken(secret.kraken_api_key, secret.kraken_api_secret)

## example usage

In [ ]:
import secret


# Create the Kraken client using your API credentials.
kraken = Kraken(
    secret.kraken_api_key,
    secret.kraken_api_secret,
)


# Get all account balances.
balance = kraken.balance()
print("Balance:")
print(balance)


# Get information about the BTC/CAD trading pair.
pair_info = kraken.pair_info("XBTCAD")
print("\nPair info:")
print(pair_info)


# Get the current BTC/CAD ticker.
# This includes bid, ask, last trade, high, low, volume, etc.
ticker = kraken.ticker("XBTCAD")
print("\nTicker:")
print(ticker)


# Get the first 10 levels of the BTC/CAD order book.
order_book = kraken.order_book("XBTCAD", count=10)
print("\nOrder book:")
print(order_book)


# Get all currently open orders.
open_orders = kraken.open_orders()
print("\nOpen orders:")
print(open_orders)


# Get previously closed orders.
closed_orders = kraken.closed_orders()
print("\nClosed orders:")
print(closed_orders)


# Get your trade history.
trades = kraken.trades()
print("\nTrades:")
print(trades)


# Validate a limit buy without actually submitting it.
# This is useful for testing volume, price, permissions, minimum sizes, etc.
test_order = kraken.buy_limit(
    pair="XBTCAD",
    volume="0.0001",
    price="10000",
    validate=True,
)

print("\nValidated order:")
print(test_order)


# Set this to True only when you actually want the examples below
# to submit real orders to Kraken.
LIVE_TRADING = False


if LIVE_TRADING:

    # Submit a real limit buy order.
    # This buys 0.0001 BTC if BTC/CAD reaches $10,000 CAD.
    order = kraken.buy_limit(
        pair="XBTCAD",
        volume="0.0001",
        price="10000",
    )

    print("\nSubmitted order:")
    print(order)


    # Get the Kraken transaction ID for the newly created order.
    txid = order["txid"][0]

    print("\nTransaction ID:")
    print(txid)


    # Query the status and details of one specific order.
    order_status = kraken.order(txid)

    print("\nOrder status:")
    print(order_status)


    # Change the price of the existing order.
    # Kraken's AmendOrder changes the existing order rather than
    # cancelling it and creating an entirely new order.
    amended = kraken.amend(
        txid,
        price="11000",
    )

    print("\nAmended order:")
    print(amended)


    # Cancel the order.
    cancelled = kraken.cancel(txid)

    print("\nCancelled order:")
    print(cancelled)


    # Submit a maker-only limit buy.
    # post_only prevents the order from immediately taking liquidity.
    maker_order = kraken.buy_limit(
        pair="XBTCAD",
        volume="0.0001",
        price="10000",
        post_only=True,
    )

    print("\nMaker order:")
    print(maker_order)


    # Submit a limit sell.
    sell_order = kraken.sell_limit(
        pair="XBTCAD",
        volume="0.0001",
        price="300000",
    )

    print("\nSell order:")
    print(sell_order)


    # Submit a market buy.
    # This executes against the best available sell orders immediately.
    market_buy = kraken.buy_market(
        pair="XBTCAD",
        volume="0.0001",
    )

    print("\nMarket buy:")
    print(market_buy)


    # Submit a market sell.
    # This executes against the best available buy orders immediately.
    market_sell = kraken.sell_market(
        pair="XBTCAD",
        volume="0.0001",
    )

    print("\nMarket sell:")
    print(market_sell)


    # Cancel every currently open order on the account.
    result = kraken.cancel_all()

    print("\nCancel all:")
    print(result)


# Enable Kraken's dead-man switch.
# If this command is not refreshed before 60 seconds expires,
# Kraken automatically cancels your open orders.
kraken.cancel_after(60)


# In a real trading loop you would call this periodically,
# for example every 20 seconds, to keep the dead-man switch alive.
kraken.cancel_after(60)


# Disable the dead-man switch when shutting the program down normally.
kraken.cancel_after(0)


# Close the underlying HTTP session when finished.
kraken.close()